In [4]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langchain_tavily import TavilySearch     
from langchain_classic import hub     
from langchain.tools import tool
import requests

In [6]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [7]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [8]:
search_tool = TavilySearch(max_results=2)

In [9]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [10]:
result = search_tool.invoke("Give me the latest news on Nigeria")
result

{'query': 'Give me the latest news on Nigeria',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.aljazeera.com/where/nigeria',
   'title': "Nigeria | Today's latest from Al Jazeera",
   'content': "Nigeria · Army frees 360 people abducted by Boko Haram in Nigeria's Borno state · Years after dropping out, women in northern Nigeria return to the classroom.",
   'score': 0.7050753,
   'raw_content': None},
  {'url': 'https://punchng.com',
   'title': 'Punch newspapers - Breaking News, Nigerian News & Top Stories',
   'content': 'Latest News · Katsina extends automatic jobs scheme to first-class graduates · LAUTECH doctors urge Makinde to disburse training fund, pay arrears.',
   'score': 0.6641271,
   'raw_content': None}],
 'response_time': 0.63,
 'request_id': '66c4ffec-25da-47b5-9c43-1510893ddce7'}

In [12]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",        
    temperature=0,
    api_key=GEMINI_API_KEY,        
    
)

response = llm.invoke("Tell me a joke about AI")
print(response.content)   # For simple string output

A human asks an AI, "Hey, tell me a joke!"

The AI processes billions of data points, analyzes comedic structures, and then replies:

"Why did the chicken cross the road? To optimize its pathfinding algorithm and minimize travel time."


In [16]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

In [17]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [18]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [19]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [20]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [21]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
Action: tavily_search
Action Input: capital of India{'query': 'capital of India', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.quora.com/Which-one-is-the-capital-of-India', 'title': 'Which one is the capital of India? - Quora', 'content': "Delhi is India's capital and national capital of India. New Delhi is the national capital of India. The capital of India is New Delhi.", 'score': 0.8800674, 'raw_content': None}, {'url': 'https://timesofindia.indiatimes.com/india/list-of-cities-that-served-as-the-capital-of-india/articleshow/111815680.cms', 'title': 'List of cities that served as the capital of India | India News', 'content': "* List of cities that served as the capital of India. Discover the list of cities that served as India's capital. This has evolved the Indian capitals often: Pataliputra during the Maurya and Gupta eras, Delhi under the Sultanate, and Agra. Calcutta as British India’s capital

In [22]:
print(response["output"])

The capital of India is New Delhi. The current weather in New Delhi is partly cloudy with a temperature of 36°C and 33% humidity.
